# 🔍 Fine-Tuning con BERT: Conceptos Teóricos

## ¿Qué es el Fine-Tuning?

El **fine-tuning** es el proceso de tomar un modelo preentrenado (como BERT, GPT, etc.) y **ajustarlo con tus propios datos etiquetados**, para que aprenda tareas específicas.

Por ejemplo:
- Tomamos `bert-base-uncased`, que fue entrenado en Wikipedia.
- Lo entrenamos con nuestros propios datos para que aprenda a clasificar comentarios positivos o negativos.

---

## ¿Por qué usar fine-tuning?

- ✅ No necesitamos entrenar desde cero (ahorra GPU, tiempo y datos).
- ✅ Aprovechamos el conocimiento previo del modelo.
- ✅ Mejora el rendimiento en tareas específicas (clasificación, NER, QA, etc.).


# 🛠️ Paso 1: Instalar librerías necesarias

In [1]:
!pip install datasets==3.5.0 transformers==4.48.3 evaluate==0.4.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 14.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

# 📚 Paso 2: Cargar un dataset de ejemplo (IMDb)

In [2]:
from datasets import load_dataset

# Cargar dataset IMDb
dataset = load_dataset("imdb")

# Mostrar ejemplo
dataset["train"][0]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

# ✂️ Paso 3: Submuestreo (opcional para entrenar más rápido)

In [3]:
# Tomamos solo 2000 ejemplos de train y 1000 de test para hacer pruebas rápidas
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset["test"].shuffle(seed=42).select(range(1000))

# 🔠 Paso 4: Cargar el tokenizer de BERT

In [4]:
from transformers import AutoTokenizer

# Usamos el modelo base de BERT uncased
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Probar el tokenizer
print(tokenizer("this movie was amazing!", padding=True, truncation=True))

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': [101, 2023, 3185, 2001, 6429, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


# 🧹 Paso 5: Tokenizar todo el dataset

In [5]:
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_train = small_train.map(tokenize_function, batched=True)
tokenized_test = small_test.map(tokenize_function, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

# ✅ Paso 6: Preparar los data loaders

In [6]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Convertimos a formato compatible con PyTorch
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# 🚀 Paso 7: Cargar el modelo BERT para clasificación

In [7]:
from transformers import AutoModelForSequenceClassification

# Cargamos BERT con una capa final para clasificación binaria (2 clases)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# ⚙️ Paso 8: Configurar los parámetros de entrenamiento

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",             # 📁 Carpeta donde se guardarán los resultados del modelo entrenado.
    evaluation_strategy="epoch",        # 📊 Estrategia de evaluación: aquí se evalúa el modelo al final de cada época.
    save_strategy="epoch",              # 💾 Guarda el modelo también al final de cada época.
    logging_dir="./logs",               # 📝 Directorio donde se almacenan los registros del entrenamiento (logs).
    per_device_train_batch_size=8,      # 🧠 Tamaño del batch (lote) por dispositivo para entrenamiento. Aquí se usan 8 ejemplos por lote.
    per_device_eval_batch_size=8,       # 🧠 Tamaño del batch por dispositivo para evaluación.
    num_train_epochs=3,                 # 🔁 Número total de épocas de entrenamiento (pasa 3 veces por todos los datos).
    weight_decay=0.01,                  # ⚖️ Aplicación de regularización L2 (weight decay) para evitar overfitting.
    logging_steps=10,                   # 📌 Número de pasos de entrenamiento entre cada log (registro en consola).
    load_best_model_at_end=True,        # 🏆 Carga automáticamente el mejor modelo evaluado al final del entrenamiento.
    save_total_limit=2                  # 🧹 Limita a 2 el número total de checkpoints guardados para ahorrar espacio.
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# 🧪 Paso 9: Definir el evaluador (función de métricas)

In [9]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, predictions)
    prec, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    return {"accuracy": acc, "precision": prec, "recall": recall, "f1": f1}

# 🏋️ Paso 10: Entrenar el modelo con Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Iniciar entrenamiento
trainer.train()

/tmp/ipython-input-10-616929469.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cmaytadatag4 (cmaytadatag4-codigo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.294800,0.317693,0.857000,0.796902,0.948770,0.866230
2,0.234700,0.501869,0.885000,0.857965,0.915984,0.886026


# 📈 Paso 11: Evaluar en el conjunto de prueba

In [ ]:
trainer.evaluate()

# PRUEBA DEL MODELO

In [ ]:
# 🔍 Inferencia manual con trainer y tokenizer
import torch

# Texto nuevo para predecir
texto = "This movie was absolutely wonderful and I loved every minute of it!"

# Tokenizamos el texto (como en entrenamiento)
inputs = tokenizer(texto, return_tensors="pt", truncation=True, padding=True)

# Mover los inputs al mismo dispositivo que el modelo
inputs = {k: v.to(model.device) for k, v in inputs.items()}


# Asegurarse de que el modelo esté en modo evaluación
model.eval()

# Desactivar cálculo de gradientes (inferencia solamente)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    pred = torch.argmax(logits, dim=1)

# Mostrar la predicción (LABEL_0 o LABEL_1)
print("Predicción:", pred.item())

# 💾 Paso 12 (opcional): Guardar el modelo y publicarlo en hugging face

In [ ]:
trainer.save_model("cmaytag4-bert-imdb-finetuned")
tokenizer.save_pretrained("cmaytag4-bert-imdb-finetuned")

In [ ]:
!huggingface-cli login

In [ ]:
# 💾 Guardar y subir el modelo a Hugging Face Hub
trainer.push_to_hub("cmaytag4-bert-imdb-finetuned")